# Part D: Evaluation and Interpretation of Findings

ZOU Rongbin (4301048) · CDS3004 Data Mining

In Part D, I evaluated the models and interpreted the findings by building upon the outputs from Students A, B, and C:

- **Student A (Part A: Data Collection and Cleaning)** prepared the cleaned group survey dataset. I cross-checked `survey_cleaned_python.xlsx` against the `Survey_Cleaned` sheet in the group's `Combined_Survey_and_UCI_Data.xlsx` workbook, and used the `UCI_Cleaned` sheet to verify the GPA labels for the higher-education test set.
- **Student B (Part B: Exploratory Analysis and Feature Preparation)** explored both the group survey and a separate Portuguese secondary-school dataset. I used `student_performance_cleaned.csv` to validate the secondary-school findings and establish baseline context for the evaluation.
- **Student C (Part C: Model Development and Comparison)** trained and compared classifiers across both datasets. For the group survey, I verified the KNN metrics using the saved scores, labelled confusion matrix, and class order from `part_c_model_development1111.ipynb`. For the UCI higher-education dataset, I validated `split.csv` and `test_predictions.csv` against the cleaned data and recalculated the test metrics directly from the exported predictions.

Place all input spreadsheets and CSV files in the same directory and set `INPUT_DIR` in the first code cell to that folder path. This notebook relies on `pandas`, `NumPy`, `openpyxl`, and `Matplotlib` to generate and display aggregate results.

The individual-habit analysis employs Bergsma's bias-corrected Cramér's V ([https://doi.org/10.1016/j.jkss.2012.10.002](https://doi.org/10.1016/j.jkss.2012.10.002)), using Pearson's chi-square as the test statistic across permutations with fixed marginal distributions. Holm's correction is applied across the four habits within each cohort. Note that these comparisons are exploratory, unadjusted for potential confounders, and sensitive to variations in question wording across surveys.


## Check Student A's cleaned survey and Student B's secondary-school analysis

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

# Set this to the folder with the project inputs if they are stored elsewhere.
INPUT_DIR = Path(os.environ.get('PART_D_INPUT_DIR', '.'))
OUTPUT_DIR = Path(os.environ.get('PART_D_OUTPUT_DIR', '.'))
LABELS = ['below 2.5', '2.5-2.99', '3.0-3.49', '3.5 above']
MODELS = ['Baseline', 'Decision Tree', 'KNN', 'SVM']

survey_path = INPUT_DIR / 'survey_cleaned_python.xlsx'
if not survey_path.is_file():
    survey_path = INPUT_DIR / 'survey_cleaned_python (1).xlsx'
survey = pd.read_excel(survey_path, sheet_name='Cleaned Survey')
workbook = INPUT_DIR / 'Combined_Survey_and_UCI_Data.xlsx'
pd.testing.assert_frame_equal(survey, pd.read_excel(workbook, sheet_name='Survey_Cleaned'))
gpa = survey['the range your Gpa grade of last semester']
hours = survey['How much time you spent on studying per week ( outside of class time)']
counts = gpa.value_counts().reindex(LABELS)
print('A/B survey rows:', len(survey), '| GPA counts:', counts.to_dict())
for name in ('0-5 hours', '16-20 hours'):
    group = gpa[hours.eq(name)]
    print(f'{name}: {(group == LABELS[-1]).sum()}/{len(group)} in the top GPA band')

secondary = pd.read_csv(INPUT_DIR / 'student_performance_cleaned.csv')
print('B secondary-school subject records:', secondary['subject'].value_counts().to_dict())
print('Pooled Pearson r, coded study time and G3:', round(secondary['studytime'].corr(secondary['G3']), 3))
print('Pooled Pearson r, absences and G3:', round(secondary['absences'].corr(secondary['G3']), 3))



A/B survey rows: 46 | GPA counts: {'below 2.5': 3, '2.5-2.99': 5, '3.0-3.49': 21, '3.5 above': 17}
0-5 hours: 5/12 in the top GPA band
16-20 hours: 4/6 in the top GPA band
B secondary-school subject records: {'Portuguese': 649, 'Math': 395}
Pooled Pearson r, coded study time and G3: 0.162
Pooled Pearson r, absences and G3: -0.046


## Recalculate survey KNN metrics from Student C's confusion matrix

In [2]:
def score_from_confusion(confusion):
    """Rows are actual bands. Columns are predicted bands in LABELS order."""
    array = confusion.to_numpy(dtype=int)
    correct = np.diag(array)
    support = array.sum(axis=1)
    predicted = array.sum(axis=0)
    recall = np.divide(correct, support, out=np.zeros(len(LABELS)), where=support > 0)
    f1 = np.divide(2 * correct, support + predicted, out=np.zeros(len(LABELS)), where=support + predicted > 0)
    return {
        'Correct': int(correct.sum()), 'n': int(array.sum()),
        'Accuracy': float(correct.sum() / array.sum()),
        'Macro F1': float(f1.mean()), 'Balanced accuracy': float(recall.mean()),
    }

# Transcribed from C's saved Survey KNN confusion matrix (comparison cell).
# The saved Survey comparison table supplies other original model scores.
survey_knn = pd.DataFrame([[0, 0, 1, 0], [0, 1, 0, 0], [0, 1, 4, 0], [0, 1, 1, 1]],
                          index=LABELS, columns=LABELS)
survey_summary = pd.DataFrame({'Training CV macro F1': [.154, .290, .313, .249],
                               'Accuracy': [.500, .500, .600, .600],
                               'Macro F1': [.167, .321, .432, .304]}, index=MODELS)
survey_score = score_from_confusion(survey_knn)
assert survey_score['n'] == 10
assert round(survey_score['Macro F1'], 3) == survey_summary.loc['KNN', 'Macro F1']
assert survey_score['Accuracy'] == survey_summary.loc['KNN', 'Accuracy']
print('C survey model comparison (saved results)')
print(survey_summary.to_string())
print('Survey KNN counts and scores recalculated from C’s matrix:', survey_score)



C survey model comparison (saved results)
               Training CV macro F1  Accuracy  Macro F1
Baseline                      0.154       0.5     0.167
Decision Tree                 0.290       0.5     0.321
KNN                           0.313       0.6     0.432
SVM                           0.249       0.6     0.304
Survey KNN counts and scores recalculated from C’s matrix: {'Correct': 6, 'n': 10, 'Accuracy': 0.6, 'Macro F1': 0.4318181818181818, 'Balanced accuracy': 0.5333333333333333}


## Verify Student C's UCI split and score the exported predictions

In [3]:
uci = pd.read_excel(workbook, sheet_name='UCI_Cleaned')
gpa_codes = {1: LABELS[0], 2: LABELS[0], 3: LABELS[1], 4: LABELS[2], 5: LABELS[3]}
uci_gpa = uci['Last_Semester_GPA'].map(gpa_codes)
split = pd.read_csv(INPUT_DIR / 'split.csv', index_col=0).sort_index()
predictions = pd.read_csv(INPUT_DIR / 'test_predictions.csv', index_col='source_row_zero_based').sort_index()
assert split.index.is_unique and split.index.tolist() == list(uci.index)
assert predictions.index.is_unique and set(split['set']) == {'train', 'test'}
test_rows = split.index[split['set'].eq('test')]
assert test_rows.tolist() == predictions.index.tolist()
pd.testing.assert_series_equal(predictions['actual'], uci_gpa.loc[test_rows], check_names=False)
training_majority = uci_gpa.loc[split.index[split['set'].eq('train')]].mode().item()
assert predictions['Baseline'].eq(training_majority).all()
assert len(test_rows) == 29 and len(uci) - len(test_rows) == 116

uci_confusions = {}
uci_scores = {}
for name in MODELS:
    assert predictions[name].isin(LABELS).all()
    confusion = pd.crosstab(predictions['actual'], predictions[name]).reindex(index=LABELS, columns=LABELS, fill_value=0)
    uci_confusions[name] = confusion
    uci_scores[name] = score_from_confusion(confusion)
uci_summary = pd.DataFrame.from_dict(uci_scores, orient='index')
uci_summary['Training CV macro F1'] = [.138, .190, .267, .209]  # C's saved training CV scores
print('C UCI held-out predictions verified against split and workbook')
print(uci_summary.round(3).to_string())
print('UCI KNN confusion matrix')
print(uci_confusions['KNN'].to_string())



C UCI held-out predictions verified against split and workbook
               Correct   n  Accuracy  Macro F1  Balanced accuracy  Training CV macro F1
Baseline            11  29     0.379     0.138              0.250                 0.138
Decision Tree       12  29     0.414     0.252              0.298                 0.190
KNN                 13  29     0.448     0.280              0.338                 0.267
SVM                 10  29     0.345     0.195              0.244                 0.209
UCI KNN confusion matrix
KNN        below 2.5  2.5-2.99  3.0-3.49  3.5 above
actual                                             
below 2.5          8         1         2          0
2.5-2.99           4         0         1          0
3.0-3.49           3         0         5          0
3.5 above          4         0         1          0


## Compare individual habits with GPA within each cohort

In [4]:
# Explore each habit separately. Keep each dataset's original response categories.
habit_columns = {
    'Weekly study time': (hours.name, 'Weekly_Study_Hours'),
    'Class attendance': ('How would you describe your class attendance last year', 'Class_Attendance'),
    'Revision timing': ('How much time do you usually spend on revision before exams or major assessments', 'Exam_Prep_Timing'),
    'Note-taking': ('note_method_clean', 'Note_Taking'),
}
PERMUTATIONS = 49999


def habit_association(feature, target, seed):
    sample = pd.concat([feature.rename('habit'), target.rename('GPA')], axis=1).dropna()
    x, options = pd.factorize(sample['habit'], sort=True)
    y = sample['GPA'].map(dict(zip(LABELS, range(len(LABELS))))).to_numpy(dtype=int)
    n = len(sample)
    r, c = len(options), len(LABELS)
    observed = np.bincount(x * c + y, minlength=r * c).reshape(r, c)
    expected = np.outer(observed.sum(axis=1), observed.sum(axis=0)) / n
    assert (expected > 0).all(), 'Check empty categories before running the permutation test.'
    chi2 = float(np.sum((observed - expected) ** 2 / expected))

    # Bergsma bias correction for Cramér’s V. Zero can occur after correction.
    phi2 = max(0.0, chi2 / n - (r - 1) * (c - 1) / (n - 1))
    corrected_dim = min(r - 1 - (r - 1) ** 2 / (n - 1), c - 1 - (c - 1) ** 2 / (n - 1))
    association = float(np.sqrt(phi2 / corrected_dim))

    # Permute GPA labels within the cohort. Both marginal distributions stay fixed.
    rng = np.random.default_rng(seed)
    extreme = 0
    for start in range(0, PERMUTATIONS, 1000):
        m = min(1000, PERMUTATIONS - start)
        permuted = rng.permuted(np.tile(y, (m, 1)), axis=1)
        keys = np.arange(m)[:, None] * (r * c) + x[None, :] * c + permuted
        tables = np.bincount(keys.ravel(), minlength=m * r * c).reshape(m, r, c)
        simulated = ((tables - expected) ** 2 / expected).sum(axis=(1, 2))
        extreme += int(np.count_nonzero(simulated >= chi2 - 1e-10))
    p = (extreme + 1) / (PERMUTATIONS + 1)
    counts = pd.DataFrame(observed, index=pd.Index(options, name=feature.name), columns=LABELS)
    return {'n': n, 'Corrected V': association, 'Permutation p': p,
            'Expected cells under 5': int((expected < 5).sum()), 'Cells': r * c}, counts


def holm(values):
    ordered = values.sort_values()
    corrected = (ordered * np.arange(len(ordered), 0, -1)).cummax().clip(upper=1)
    return corrected.reindex(values.index)


def study_habits(survey_rows, survey_labels):
    records, cross_tabs = [], {}
    for j, (habit, (survey_column, uci_column)) in enumerate(habit_columns.items()):
        cohorts = [
            ('Group survey', survey_rows[survey_column], survey_labels),
            ('UCI higher education', uci[uci_column], uci_gpa),
        ]
        for i, (cohort, values, target) in enumerate(cohorts):
            result, counts = habit_association(values, target, seed=20261118 + 2 * j + i)
            records.append({'Cohort': cohort, 'Habit': habit, **result})
            cross_tabs[(cohort, habit)] = counts
    table = pd.DataFrame(records).set_index(['Cohort', 'Habit'])
    table['Holm p'] = holm(table['Permutation p'])
    return table, cross_tabs


habit_results, habit_tables = study_habits(survey, gpa)
print('Four separate habits tested within each sample. Holm adjustment covers all eight tests.')
print(habit_results.round(3).to_string())

# Recheck conclusions after keeping only the first record in the matching response pair.
keep = ~survey.drop(columns='Timestamp').duplicated(keep='first')
sensitivity, _ = study_habits(survey.loc[keep], gpa.loc[keep])
print('\nMatching-answer sensitivity. Survey n =', int(keep.sum()))
print(sensitivity.loc['Group survey', ['Corrected V', 'Holm p']].round(3).to_string())

uci_habit_labels = {
    'Weekly study time': {1: 'None', 2: '<5 hours', 3: '6–10 hours', 4: '11–20 hours', 5: '>20 hours'},
    'Class attendance': {1: 'Always', 2: 'Sometimes', 3: 'Never'},
    'Revision timing': {1: 'Close to the exam', 2: 'Regularly', 3: 'Never'},
    'Note-taking': {1: 'Never', 2: 'Sometimes', 3: 'Always'},
}
print('\nGPA band counts by habit and cohort')
for (cohort, habit), tab in habit_tables.items():
    if cohort == 'UCI higher education':
        tab = tab.rename(index=uci_habit_labels[habit])
    print(f'\n{cohort} | {habit}')
    print(tab.assign(Total=tab.sum(axis=1)).to_string())



Four separate habits tested within each sample. Holm adjustment covers all eight tests.
                                          n  Corrected V  Permutation p  Expected cells under 5  Cells  Holm p
Cohort               Habit                                                                                    
Group survey         Weekly study time   46        0.000          0.571                      23     24   1.000
UCI higher education Weekly study time  145        0.078          0.260                       8     20   0.947
Group survey         Class attendance    46        0.298          0.010                      13     16   0.077
UCI higher education Class attendance   145        0.157          0.090                       0      8   0.541
Group survey         Revision timing     46        0.240          0.111                      18     20   0.554
UCI higher education Revision timing    145        0.153          0.039                       6     12   0.275
Group survey         Not

## Plot the original KNN confusion matrices

In [5]:
import matplotlib.pyplot as plt

uci_knn = uci_confusions['KNN']
fig, axes = plt.subplots(1, 2, figsize=(11.4, 5.2))
fig.suptitle('KNN misclassified all test cases in two UCI GPA bands',
             x=.07, y=.98, ha='left', fontsize=17, fontweight='bold', color='#203344')
ticks = ['<2.5', '2.5–2.99', '3.0–3.49', 'Top band']
panels = [(survey_knn, 'Group survey', survey_score), (uci_knn, 'UCI higher education', uci_scores['KNN'])]
for ax, (matrix, name, score) in zip(axes, panels):
    values = matrix.to_numpy()
    ax.imshow(values, cmap='Blues', vmin=0, vmax=8)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, str(values[i, j]), ha='center', va='center', fontsize=13,
                    color='white' if values[i, j] >= 5 else '#203344',
                    fontweight='bold' if i == j else 'normal')
    ax.set_xticks(range(4), ticks, fontsize=9)
    ax.set_yticks(range(4), ticks, fontsize=9)
    ax.tick_params(length=0, pad=6)
    ax.set_xlabel('Predicted GPA band', labelpad=8, fontsize=10)
    ax.set_ylabel('Actual GPA band', labelpad=8, fontsize=10)
    ax.set_title(f"{name}   |   {score['Correct']}/{score['n']} correct", loc='left', pad=11,
                 fontsize=12, fontweight='bold', color='#203344')
    for spine in ax.spines.values():
        spine.set_color('#C4D2DF')
fig.text(.07, .045, 'Counts from Part C’s original holdout results. The diagonal shows correct predictions.',
         fontsize=9, color='#425366')
fig.subplots_adjust(left=.08, right=.96, bottom=.19, top=.82, wspace=.36)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_DIR / 'D_KNN_Confusion_Matrices.png', dpi=220, facecolor='white')
plt.close(fig)
print('Saved D_KNN_Confusion_Matrices.png')


Saved D_KNN_Confusion_Matrices.png
